In [2]:
from pathlib import Path
import json


CHUNK_DIR = Path("../../../data/chunks/hana")
RESULT_A = Path.home() / "pilot_work/out8_hana50"
RESULT_B = Path.home() / "pilot_work/out9_hana50"
CHUNK_ID = "13a894c055fd6895-p0011-c01"
GROUPS = ("benefits", "conditions", "limits", "targets", "relations")

# def load_results(path):
#     return {r["chunk_id"]: r for f in path.glob("*.result.json")
#             if (r := json.loads(f.read_text(encoding="utf-8"))).get("chunk_id")}

# a_rows, b_rows = load_results(RESULT_A), load_results(RESULT_B)


In [ ]:
# Cell 1 — result.json metrics

def metrics(rows):
    count = len(rows)
    successes = sum(r.get("extraction") is not None and not any(f.get("code") == "schema_validation_failure" for f in r.get("failures", [])) for r in rows.values())
    empty = sum(any(f.get("code") == "empty_result" for f in r.get("failures", [])) for r in rows.values())
    schema = sum(any(f.get("code") == "schema_validation_failure" for f in r.get("failures", [])) for r in rows.values())
    items = sum(len((r.get("extraction") or {}).get(g) or []) for r in rows.values() for g in GROUPS[:-1])
    relations = sum(len((r.get("extraction") or {}).get("relations") or []) for r in rows.values())
    evidence = sum(len(item.get("evidence") or []) for r in rows.values()
                   for g in GROUPS for item in ((r.get("extraction") or {}).get(g) or []))
    return {"chunks": count, "success chunks": successes, "empty_result": empty,
            "schema violations": schema, "items (4 entity groups)": items, "relations": relations, "evidence": evidence}

metric_rows = metrics(a_rows), metrics(b_rows)
headers = ["metric", "out8", "out9"]
print("| " + " | ".join(headers) + " |")
print("|" + "|".join(["---"] * len(headers)) + "|")
for key in metric_rows[0]:
    print(f"| {key} | {metric_rows[0][key]} | {metric_rows[1][key]} |")


In [ ]:
# Cell 2 — chunks empty in both runs, with counts only
from payment_card.file_pipeline.extraction.report.table_density_report import _chunk_index, density_score, classify_density

chunk_index = _chunk_index(CHUNK_DIR)
ids = sorted(a_rows.keys() & b_rows.keys())
def is_empty(row):
    return any(f.get("code") == "empty_result" for f in row.get("failures", []))
empty_both = sorted(cid for cid in ids if is_empty(a_rows[cid]) and is_empty(b_rows[cid]))
scores = {cid: density_score(chunk_index[cid].get("text", "")) for cid in ids if cid in chunk_index}
labels = dict(zip(sorted(scores), classify_density([scores[cid] for cid in sorted(scores)])))
terms = ["%", "할인", "적립", "캐시백", "원", "포인트", "마일리지", "이용"]
summary = []
for cid in empty_both:
    chunk = chunk_index[cid]
    text = chunk.get("text", "")
    row = {"chunk_id": cid, "length": len(text), "table_density": labels.get(cid, "unknown"),
           "page_no": chunk.get("page_no")}
    row.update({f"count:{term}": text.count(term) for term in terms})
    summary.append(row)
headers = list(summary[0]) if summary else ["chunk_id"]
print("| " + " | ".join(headers) + " |")
print("|" + "|".join(["---"] * len(headers)) + "|")
for row in summary:
    print("| " + " | ".join(str(row[h]) for h in headers) + " |")


In [ ]:
# Cell 3 — view one selected chunk on screen only; do not save notebook outputs
selected = chunk_index[CHUNK_ID]
print(selected.get("text", ""))


In [ ]:
# Cell 4 — side-by-side item counts for the selected chunk

def counts(row):
    extraction = row.get("extraction") or {}
    return {group: len(extraction.get(group) or []) for group in GROUPS}
left, right = counts(a_rows[CHUNK_ID]), counts(b_rows[CHUNK_ID])
print("| group | out8 | out9 |\n|---|---:|---:|")
for group in GROUPS:
    print(f"| {group} | {left[group]} | {right[group]} |")
